# Baseline model comparison

Train Logistic Regression and Linear SVM on the training split and compare their performance on the internal validation split.

In [1]:
from pathlib import Path
import sys
import time

import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.train import build_logistic_regression_pipeline

In [2]:
TRAIN_PATH = PROJECT_ROOT / "data" / "processed" / "train_clean.csv"
VALIDATION_PATH = PROJECT_ROOT / "data" / "processed" / "validation_clean.csv"

train_df = pd.read_csv(TRAIN_PATH)
validation_df = pd.read_csv(VALIDATION_PATH)

print("Training shape:", train_df.shape)
print("Validation shape:", validation_df.shape)

display(train_df.head())

Training shape: (50301, 4)
Validation shape: (12682, 4)


,tweet_id,entity,sentiment,text
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...
1,2401,Borderlands,Positive,i am coming to the borders and i will kill you...
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
4,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...


In [3]:
X_train = train_df["text"]
y_train = train_df["sentiment"]

X_validation = validation_df["text"]
y_validation = validation_df["sentiment"]

print("Training samples:", len(X_train))
print("Validation samples:", len(X_validation))

Training samples: 50301
Validation samples: 12682


In [4]:
baseline_model = build_logistic_regression_pipeline()

baseline_model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidf', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",False
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",<function nor...0026E632CC220>
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",<function tok...0026E632ECD60>
,"token_pattern token_pattern: str, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp selects tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",None
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"max_df max_df: float or int, default=1.0When building the vocabulary ignore terms that have a documentfrequency strictly higher than the given threshold (corpus-specificstop words).If float in range [0.0, 1.0], the parameter represents a proportion ofdocuments, integer absolute counts.This parameter is ignored if vocabulary is not None.",0.98
,"min_df min_df: float or int, default=1When building the vocabulary ignore terms that have a documentfrequency strictly lower than the given threshold. This value is alsocalled cut-off in the literature.If float in range of [0.0, 1.0], the parameter represents a proportionof documents, integer absolute counts.This parameter is ignored 

In [5]:
print(baseline_model.named_steps.keys())

dict_keys(['tfidf', 'classifier'])


In [6]:
start_time = time.perf_counter()

baseline_model.fit(X_train, y_train)

training_time = time.perf_counter() - start_time

print(f"Training time: {training_time:.2f} seconds")

Training time: 15.28 seconds


In [7]:
start_time = time.perf_counter()

validation_predictions = baseline_model.predict(X_validation)

prediction_time = time.perf_counter() - start_time

print(f"Prediction time: {prediction_time:.4f} seconds")

Prediction time: 1.7348 seconds


In [8]:
accuracy = accuracy_score(
    y_validation,
    validation_predictions,
)

macro_precision, macro_recall, macro_f1, _ = (
    precision_recall_fscore_support(
        y_validation,
        validation_predictions,
        average="macro",
        zero_division=0,
    )
)

weighted_precision, weighted_recall, weighted_f1, _ = (
    precision_recall_fscore_support(
        y_validation,
        validation_predictions,
        average="weighted",
        zero_division=0,
    )
)

baseline_metrics = pd.DataFrame(
    {
        "metric": [
            "accuracy",
            "macro_precision",
            "macro_recall",
            "macro_f1",
            "weighted_precision",
            "weighted_recall",
            "weighted_f1",
            "training_time_seconds",
            "prediction_time_seconds",
        ],
        "value": [
            accuracy,
            macro_precision,
            macro_recall,
            macro_f1,
            weighted_precision,
            weighted_recall,
            weighted_f1,
            training_time,
            prediction_time,
        ],
    }
)

baseline_metrics

,metric,value
0,accuracy,0.573175
1,macro_precision,0.547069
2,macro_recall,0.539104
3,macro_f1,0.536343
4,weighted_precision,0.560675
5,weighted_recall,0.573175
6,weighted_f1,0.561306
7,training_time_seconds,15.284238
8,prediction_time_seconds,1.734795


In [9]:
report = classification_report(
    y_validation,
    validation_predictions,
    zero_division=0,
)

print(report)

              precision    recall  f1-score   support

  Irrelevant       0.43      0.26      0.32      2228
    Negative       0.60      0.74      0.66      3898
     Neutral       0.55      0.54      0.55      3072
    Positive       0.60      0.62      0.61      3484

    accuracy                           0.57     12682
   macro avg       0.55      0.54      0.54     12682
weighted avg       0.56      0.57      0.56     12682



In [10]:
labels = sorted(y_validation.unique())

confusion = confusion_matrix(
    y_validation,
    validation_predictions,
    labels=labels,
)

confusion_df = pd.DataFrame(
    confusion,
    index=[f"actual_{label}" for label in labels],
    columns=[f"predicted_{label}" for label in labels],
)

confusion_df

,predicted_Irrelevant,predicted_Negative,predicted_Neutral,predicted_Positive
actual_Irrelevant,578,629,473,548
actual_Negative,229,2871,413,385
actual_Neutral,249,679,1663,481
actual_Positive,301,576,450,2157


In [11]:
prediction_results = validation_df[
    ["tweet_id", "entity", "text", "sentiment"]
].copy()

prediction_results["predicted_sentiment"] = (
    validation_predictions
)

prediction_results["correct"] = (
    prediction_results["sentiment"]
    == prediction_results["predicted_sentiment"]
)

mistakes = prediction_results[
    ~prediction_results["correct"]
]

print("Correct predictions:", prediction_results["correct"].sum())
print("Incorrect predictions:", len(mistakes))

display(mistakes.head(20))

Correct predictions: 7269
Incorrect predictions: 5413


,tweet_id,entity,text,sentiment,predicted_sentiment,correct
16,2413,Borderlands,imma probably got some video tps in a bit. tha...,Positive,Neutral,False
17,2415,Borderlands,fuck yessssssss .,Positive,Negative,False
19,2415,Borderlands,fuck yessssssss.,Positive,Negative,False
20,2415,Borderlands,fuck yessssssss,Positive,Negative,False
21,2415,Borderlands,a fuck... yessssssss.,Positive,Negative,False
22,2415,Borderlands,fuck you.,Positive,Negative,False
29,2424,Borderlands,how the hell are we into halloween month alrea...,Irrelevant,Negative,False
30,2424,Borderlands,how the hell are we already into halloween mon...,Irrelevant,Negative,False
31,2424,Borderlands,how the hell are we already in halloween month?!.,Irrelevant,Negative,False
32,2424,Borderlands,how the hell are march into halloween month al...,Irrelevant,Negative,False


In [12]:
RESULTS_DIR = PROJECT_ROOT / "reports" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [13]:
report_dict = classification_report(
    y_validation,
    validation_predictions,
    output_dict=True,
    zero_division=0,
)

report_df = pd.DataFrame(report_dict).transpose()

report_df.to_csv(
    RESULTS_DIR / "baseline_logistic_regression_report.csv"
)

In [14]:
from pathlib import Path
import sys
import time

import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.train import build_svm_pipeline

In [15]:
TRAIN_PATH = PROJECT_ROOT / "data" / "processed" / "train_clean.csv"
VALIDATION_PATH = PROJECT_ROOT / "data" / "processed" / "validation_clean.csv"

train_df = pd.read_csv(TRAIN_PATH)
validation_df = pd.read_csv(VALIDATION_PATH)

print("Training shape:", train_df.shape)
print("Validation shape:", validation_df.shape)

Training shape: (50301, 4)
Validation shape: (12682, 4)


In [16]:
X_train = train_df["text"]
y_train = train_df["sentiment"]

X_validation = validation_df["text"]
y_validation = validation_df["sentiment"]

print("Training samples:", len(X_train))
print("Validation samples:", len(X_validation))

Training samples: 50301
Validation samples: 12682


In [17]:
svm_model = build_svm_pipeline()

svm_model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidf', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",False
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",<function nor...0026E632CC220>
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",<function tok...0026E632ECD60>
,"token_pattern token_pattern: str, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp selects tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",None
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"max_df max_df: float or int, default=1.0When building the vocabulary ignore terms that have a documentfrequency strictly higher than the given threshold (corpus-specificstop words).If float in range [0.0, 1.0], the parameter represents a proportion ofdocuments, integer absolute counts.This parameter is ignored if vocabulary is not None.",0.98
,"min_df min_df: float or int, default=1When building the vocabulary ignore terms that have a documentfrequency strictly lower than the given threshold. This value is alsocalled cut-off in the literature.If float in range of [0.0, 1.0], the parameter represents a proportionof documents, integer absolute counts.This parameter is ignored 

In [18]:
start_time = time.perf_counter()

svm_model.fit(X_train, y_train)

training_time = time.perf_counter() - start_time

print(f"Training time: {training_time:.2f} seconds")

Training time: 9.58 seconds


In [19]:
start_time = time.perf_counter()

validation_predictions = svm_model.predict(X_validation)

prediction_time = time.perf_counter() - start_time

print(f"Prediction time: {prediction_time:.4f} seconds")

Prediction time: 1.6811 seconds


In [20]:
accuracy = accuracy_score(
    y_validation,
    validation_predictions,
)

macro_precision, macro_recall, macro_f1, _ = (
    precision_recall_fscore_support(
        y_validation,
        validation_predictions,
        average="macro",
        zero_division=0,
    )
)

weighted_precision, weighted_recall, weighted_f1, _ = (
    precision_recall_fscore_support(
        y_validation,
        validation_predictions,
        average="weighted",
        zero_division=0,
    )
)

svm_metrics = pd.DataFrame(
    {
        "metric": [
            "accuracy",
            "macro_precision",
            "macro_recall",
            "macro_f1",
            "weighted_precision",
            "weighted_recall",
            "weighted_f1",
            "training_time_seconds",
            "prediction_time_seconds",
        ],
        "value": [
            accuracy,
            macro_precision,
            macro_recall,
            macro_f1,
            weighted_precision,
            weighted_recall,
            weighted_f1,
            training_time,
            prediction_time,
        ],
    }
)

svm_metrics

,metric,value
0,accuracy,0.545340
1,macro_precision,0.518786
2,macro_recall,0.514926
3,macro_f1,0.513174
4,weighted_precision,0.534098
5,weighted_recall,0.545340
6,weighted_f1,0.536523
7,training_time_seconds,9.582194
8,prediction_time_seconds,1.681124


In [21]:
report = classification_report(
    y_validation,
    validation_predictions,
    zero_division=0,
)

print(report)

              precision    recall  f1-score   support

  Irrelevant       0.39      0.27      0.32      2228
    Negative       0.59      0.70      0.64      3898
     Neutral       0.53      0.51      0.52      3072
    Positive       0.57      0.58      0.57      3484

    accuracy                           0.55     12682
   macro avg       0.52      0.51      0.51     12682
weighted avg       0.53      0.55      0.54     12682



In [22]:
labels = sorted(y_validation.unique())

confusion = confusion_matrix(
    y_validation,
    validation_predictions,
    labels=labels,
)

confusion_df = pd.DataFrame(
    confusion,
    index=[f"actual_{label}" for label in labels],
    columns=[f"predicted_{label}" for label in labels],
)

confusion_df

,predicted_Irrelevant,predicted_Negative,predicted_Neutral,predicted_Positive
actual_Irrelevant,605,604,447,572
actual_Negative,278,2733,453,434
actual_Neutral,339,657,1560,516
actual_Positive,344,615,507,2018


In [23]:
prediction_results = validation_df[
    ["tweet_id", "entity", "text", "sentiment"]
].copy()

prediction_results["predicted_sentiment"] = (
    validation_predictions
)

prediction_results["correct"] = (
    prediction_results["sentiment"]
    == prediction_results["predicted_sentiment"]
)

mistakes = prediction_results[
    ~prediction_results["correct"]
]

print("Correct predictions:", prediction_results["correct"].sum())
print("Incorrect predictions:", len(mistakes))

display(mistakes.head(20))

Correct predictions:

 6916
Incorrect predictions: 5766


,tweet_id,entity,text,sentiment,predicted_sentiment,correct
16,2413,Borderlands,imma probably got some video tps in a bit. tha...,Positive,Neutral,False
17,2415,Borderlands,fuck yessssssss .,Positive,Negative,False
19,2415,Borderlands,fuck yessssssss.,Positive,Negative,False
20,2415,Borderlands,fuck yessssssss,Positive,Negative,False
21,2415,Borderlands,a fuck... yessssssss.,Positive,Negative,False
22,2415,Borderlands,fuck you.,Positive,Negative,False
30,2424,Borderlands,how the hell are we already into halloween mon...,Irrelevant,Negative,False
31,2424,Borderlands,how the hell are we already in halloween month?!.,Irrelevant,Negative,False
39,2455,Borderlands,big hearty congratulations to my families on y...,Positive,Irrelevant,False
41,2455,Borderlands,big heartfelt congratulations to my @ playapex...,Positive,Negative,False


In [24]:
RESULTS_DIR = PROJECT_ROOT / "reports" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

report_dict = classification_report(
    y_validation,
    validation_predictions,
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report_dict).transpose()
report_df.to_csv(RESULTS_DIR / "baseline_svm_report.csv")

The saved report contains the per-class validation metrics. Individual predictions remain in the notebook and are not stored as a large result file.

# Baseline comparison

In [25]:
logistic_values = baseline_metrics.set_index("metric")["value"]
svm_values = svm_metrics.set_index("metric")["value"]

comparison = pd.DataFrame([
    {
        "model": "Baseline Logistic Regression",
        "accuracy": logistic_values["accuracy"],
        "macro_precision": logistic_values["macro_precision"],
        "macro_recall": logistic_values["macro_recall"],
        "macro_f1": logistic_values["macro_f1"],
        "weighted_f1": logistic_values["weighted_f1"],
        "training_or_search_time_seconds": logistic_values["training_time_seconds"],
    },
    {
        "model": "Baseline Linear SVM",
        "accuracy": svm_values["accuracy"],
        "macro_precision": svm_values["macro_precision"],
        "macro_recall": svm_values["macro_recall"],
        "macro_f1": svm_values["macro_f1"],
        "weighted_f1": svm_values["weighted_f1"],
        "training_or_search_time_seconds": svm_values["training_time_seconds"],
    },
])

comparison

,model,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,training_or_search_time_seconds
0,Baseline Logistic Regression,0.573175,0.547069,0.539104,0.536343,0.561306,15.284238
1,Baseline Linear SVM,0.545340,0.518786,0.514926,0.513174,0.536523,9.582194


In [26]:
comparison.to_csv(
    PROJECT_ROOT / "reports" / "results" / "baseline_model_comparison.csv",
    index=False,
)

The supplied test split remains untouched during baseline comparison and hyperparameter selection.